# Reproduce: Barnard's Star: refusal of a saturated star

Two modes, set in the next cell:

* `MODE = "offline"` (default): never touches the network. Recomputes from the FITS fixtures in
  `backend/tests/fixtures` where the case has them, replays the published analysis from cutouts already in
  `data/cache`, and checks every cached file against the sha256 in the bundle manifest.
* `MODE = "online"`: re-downloads from IRSA (and reruns the analysis live). Slow; a checksum that changed is
  reported as DRIFT, i.e. the archive reprocessed the product after we published.

Numbers printed as "published" come from `submission/evidence`; nothing here edits them.

In [1]:
CASE = "barnard"
MODE = "offline"          # "offline" or "online"
MAX_ONLINE_ITEMS = 5      # archive items to re-download in online mode (None = all)
FULL_REPLAY = True        # rerun the whole analysis (offline: from data/cache only)

In [2]:
import json, sys, zipfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend" / "hipparchus").exists())
sys.path.insert(0, str(ROOT / "backend"))
from hipparchus import bundle

spec = bundle.CASES[CASE]
ev = json.loads((bundle.EVIDENCE / spec["evidence"]).read_text())
zpath = bundle.BUNDLE_DIR / f"{CASE}.zip"
man = json.loads(zipfile.ZipFile(zpath).read(f"{CASE}/manifest.json"))
print(spec["title"])
print("bundle", zpath.relative_to(ROOT), "| evidence sha256", man["evidence_files"][0]["sha256"][:16], "...")
print("datasets", [d["doi"] for d in man["datasets"]], "| cutouts", len(man["cutouts"]),
      "| archive queries", len(man["archive_queries"]))

Barnard's Star: the tool refuses a saturated star
bundle submission/bundles/barnard.zip | evidence sha256 f7fdc32c09ce0c78 ...
datasets ['10.26131/IRSA662', '10.26131/IRSA678'] | cutouts 58 | archive queries 2


In [3]:
comps = ev["comparisons"]
print(f"published: {ev['n_epochs']} epochs, {ev['n_usable']} usable, {len(comps)} valid pairs, "
      f"{sum(c['changed'] for c in comps)} over {ev['config']['detection_sigma']} sigma")
print("refused pairs:", ev["refused_pairs"])

published: 58 epochs, 0 usable, 0 valid pairs, 0 over 5.0 sigma
refused pairs: {'wavelength_mismatch': 1579, 'flagged_pixels': 1653, 'bad_variance': 57, 'nan_pixels': 57}


In [4]:
# Checksums: cache in offline mode, the archive itself in online mode.
r = bundle.verify(zpath, network=(MODE == "online"), limit=MAX_ONLINE_ITEMS if MODE == "online" else None)
print(json.dumps(r["summary"]))
for i in r["items"]:
    if i["status"] in ("DRIFT", "unreachable"):
        print(i["status"], i["url"][:110], i.get("note", i.get("error", "")))
print("bundle OK" if r["ok"] else "bundle NOT OK")

{"bundle_files": {"match": 12}, "data": {"not_cached": 2, "match": 58}, "source_files": {"match": 15}}
bundle OK


In [5]:
# Offline fixture: one real Barnard's Star cutout (EPSF extension stripped).
from hipparchus import quality
FIX = ROOT / "backend" / "tests" / "fixtures"
prov = json.loads((FIX / "fixtures_provenance.json").read_text())
e = quality.load(str(FIX / "barnard_d6.fits"), prov["barnard_d6"]["obs_id"], ev["ra"], ev["dec"])
pub = next(x for x in ev["epochs"] if x["obs_id"] == e.obs_id)
print("recomputed: usable", e.usable, [i.message for i in e.issues])
print("published : usable", pub["usable"], [i["message"] for i in pub["issues"]])

recomputed: usable False ['Pixels on the target are flagged: BLOOM (12 px), OVERFLOW (2 px), SUR_ERROR (2 px), TRANSIENT (10 px).']
published : usable False ['Pixels on the target are flagged: BLOOM (12 px), OVERFLOW (2 px), SUR_ERROR (2 px), TRANSIENT (10 px).']


In [6]:
# Whole analysis again. Offline: the saved archive listing and cached cutouts only.
from hipparchus import archive, cutouts, pipeline
if FULL_REPLAY:
    saved_query, saved_fetch = archive.query, cutouts.fetch
    if MODE == "offline":
        listing = json.loads(archive._manifest_path(ev["ra"], ev["dec"]).read_text())
        archive.query = lambda ra, dec, **k: listing
        def offline_fetch(url, ra, dec, size=cutouts.DEFAULT_SIZE_DEG, **k):
            if not (cutouts.CACHE_DIR / f"{cutouts._key(url, ra, dec, size)}.fits").exists():
                raise RuntimeError("not in data/cache (offline)")
            return saved_fetch(url, ra, dec, size, **k)
        cutouts.fetch = offline_fetch
    try:
        res = pipeline.analyze(ev["ra"], ev["dec"], ev["detector"])
    finally:
        archive.query, cutouts.fetch = saved_query, saved_fetch
    key = lambda c: (c["a"], c["b"])
    new = {key(c): c for c in res["comparisons"]}
    old = {key(c): c for c in comps}
    print("epochs", res["n_epochs"], "vs", ev["n_epochs"], "| usable", res["n_usable"], "vs", ev["n_usable"])
    print("same valid pairs:", set(new) == set(old), "| same verdicts:", all(new[k]["changed"] == old[k]["changed"] for k in old if k in new))
    if old and set(new) == set(old):
        print("max |significance difference|:", max(abs(new[k]["significance"] - old[k]["significance"]) for k in old))
    print("refused pairs", dict(res["refused_pairs"]), "| failed downloads", len(res["failed_downloads"]))

epochs 58 vs 58 | usable 0 vs 0
same valid pairs: True | same verdicts: True
refused pairs {'wavelength_mismatch': 1579, 'flagged_pixels': 1653, 'nan_pixels': 57, 'bad_variance': 57} | failed downloads 0
